# Welcome to Civilization — EDA Notebook

This notebook is a **thin wrapper** over the EDA scripts and src/wtc modules.
All heavy logic lives in the scripts; this notebook exists for interactive exploration.

**Prerequisites**: Run the following from the project root first:
```bash
make setup
make download
make preprocess
make eda
```


In [ ]:
import sys
from pathlib import Path

# Ensure project root is in path (works if launched from root or notebooks/)
cur_dir = Path().resolve()
for p_dir in [cur_dir, cur_dir.parent]:
    if (p_dir / 'src' / 'wtc').exists() and str(p_dir / 'src') not in sys.path:
        sys.path.insert(0, str(p_dir / 'src'))
        break

# Verify
from wtc.config import CFG, LABELS, PROJECT_ROOT
print(f'Project root: {PROJECT_ROOT}')
print(f'Labels: {LABELS}')
print(f'Seed: {CFG["seed"]}')

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from wtc.data import load_processed, load_split, load_split_manifest
from wtc.utils import set_seed

set_seed(CFG['seed'])
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)

# Load data
processed = load_processed()
train_df  = load_split('train')
val_df    = load_split('val')
test_df   = load_split('test')
manifest  = load_split_manifest()

print(f'Total processed rows: {len(processed):,}')
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

In [ ]:
# Load pre-computed EDA findings
findings_path = PROJECT_ROOT / 'reports/eda/eda_findings.json'
with open(findings_path) as f:
    findings = json.load(f)
print('EDA findings keys:', list(findings.keys()))

In [ ]:
# --- Label distribution overview ---
a = findings['a_label_distribution']
counts_s = pd.Series(a['counts'])
rates_s  = pd.Series(a['rates'])
pw_s     = pd.Series(a['pos_weight'])

summary = pd.DataFrame({'count': counts_s, 'rate': rates_s, 'pos_weight': pw_s})
summary

In [ ]:
# --- Display pre-generated figures ---
from IPython.display import Image, display
fig_dir = PROJECT_ROOT / 'reports/eda/figures'

for fig_name in sorted(fig_dir.glob('*.png')):
    print(f'\n### {fig_name.stem}')
    display(Image(str(fig_name), width=800))

In [ ]:
# --- Quick interactive exploration ---
# Example: token length distribution from findings
e = findings.get('e_tokenized_length', {})
if 'percentiles' in e:
    print('Token length percentiles:')
    for pct, val in e['percentiles'].items():
        print(f'  {pct}th: {val:.0f} tokens')

In [ ]:
# --- Re-run any individual EDA function for interactive exploration ---
from wtc.eda import label_distribution
from pathlib import Path
import tempfile

# Example: re-run label distribution on val split
with tempfile.TemporaryDirectory() as tmp:
    result = label_distribution(val_df, LABELS, Path(tmp))
    print('Val label distribution:')
    for lbl in LABELS:
        print(f'  {lbl}: {result["counts"][lbl]:,} ({result["rates"][lbl]:.3%})')